# Taller Computacional: Sistemas de Ecuaciones con NumPy
## Álgebra Lineal — 2026

En este taller aprenderás a:

1. Representar **matrices** usando la librería NumPy
2. Calcular el **rango** de una matriz con `np.linalg.matrix_rank()`
3. Determinar el **tipo de solución** de un sistema de ecuaciones usando el rango
4. **Resolver** sistemas de ecuaciones lineales con `np.linalg.solve()` y `np.linalg.lstsq()`

> **Nota:** Para ejecutar una celda presiona `Shift + Enter`.

In [32]:
import numpy as np
print("NumPy versión:", np.__version__)

NumPy versión: 1.26.4


---
## Parte 1: ¿Cómo escribir una matriz en NumPy?

Una **matriz** es un arreglo rectangular de números organizados en filas y columnas.
En NumPy se crea con `np.array()`, pasando una **lista de listas** donde cada lista
interna representa una **fila**.

**Sintaxis:**
```python
A = np.array([[a11, a12, ...],
              [a21, a22, ...],
              ...])
```

Por ejemplo, la siguiente matriz de $3 \times 3$:

$$A = \begin{pmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{pmatrix}$$

se escribe así:

In [33]:
import numpy as np

# Matriz 3×3
A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]])

print("Matriz A (3×3):")
print(A)
print()
print("Dimensiones (filas × columnas):", A.shape)
print("Número de filas    :", A.shape[0])
print("Número de columnas :", A.shape[1])

Matriz A (3×3):
[[ 2  1 -1]
 [-3 -1  2]
 [-2  1  2]]

Dimensiones (filas × columnas): (3, 3)
Número de filas    : 3
Número de columnas : 3


### Acceder a elementos, filas y columnas

Los índices en NumPy **empiezan en 0**.

| Operación | Sintaxis |
|-----------|----------|
| Elemento fila `i`, columna `j` | `A[i, j]` |
| Fila completa `i` | `A[i]`  ó  `A[i, :]` |
| Columna completa `j` | `A[:, j]` |
| Transpuesta $A^T$ | `A.T` |

In [34]:
A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]])

# Elementos individuales
print("A[0, 0] =", A[0, 0])   # fila 0, columna 0 → 2
print("A[1, 2] =", A[1, 2])   # fila 1, columna 2 → 2
print("A[2, 1] =", A[2, 1])   # fila 2, columna 1 → 1

# Filas y columnas completas
print("\nPrimera fila    :", A[0])
print("Segunda columna :", A[:, 1])

# Transpuesta
print("\nTranspuesta de A:")
print(A.T)

A[0, 0] = 2
A[1, 2] = 2
A[2, 1] = 1

Primera fila    : [ 2  1 -1]
Segunda columna : [ 1 -1  1]

Transpuesta de A:
[[ 2 -3 -2]
 [ 1 -1  1]
 [-1  2  2]]


### Matrices especiales útiles

| Función NumPy | Descripción |
|---------------|-------------|
| `np.eye(n)` | Matriz **identidad** $n \times n$ |
| `np.zeros((m, n))` | Matriz de **ceros** $m \times n$ |
| `np.ones((m, n))` | Matriz de **unos** $m \times n$ |
| `np.column_stack([A, b])` | **Apila** $A$ y $b$ en columnas → útil para $[A \mid b]$ |

In [35]:
# Matriz identidad 4×4
print("Identidad 4×4:")
print(np.eye(4))

# Construir la matriz aumentada [A | b]
A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]])
b = np.array([8, -11, -3])

Ab = np.column_stack([A, b])
print("\nMatriz aumentada [A | b]:")
print(Ab)

Identidad 4×4:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

Matriz aumentada [A | b]:
[[  2   1  -1   8]
 [ -3  -1   2 -11]
 [ -2   1   2  -3]]


### Matrices rectangulares

Las matrices no tienen por qué ser cuadradas.
Una matriz $m \times n$ con $m \neq n$ también se crea con `np.array()`:

In [36]:
# Matriz 2×4
B = np.array([[ 1, -2,  3, -4],
              [ 5,  6, -7,  8]])
print("Matriz 2×4:")
print(B)
print("Forma:", B.shape)

# Matriz 5×4 (cinco filas, cuatro columnas)
C = np.array([[1, 1, 0, 0],
              [0, 1, 1, 0],
              [0, 0, 1, 1],
              [1, 0, 0, 1],
              [1, 1, 1, 1]])
print("\nMatriz 5×4:")
print(C)
print("Forma: filas, columnas", C.shape)

Matriz 2×4:
[[ 1 -2  3 -4]
 [ 5  6 -7  8]]
Forma: (2, 4)

Matriz 5×4:
[[1 1 0 0]
 [0 1 1 0]
 [0 0 1 1]
 [1 0 0 1]
 [1 1 1 1]]
Forma: filas, columnas (5, 4)


---
## Parte 2: Rango de una Matriz

El **rango** de una matriz $A$ (notación: $\text{rango}(A)$ ó $\text{rank}(A)$) es
el número máximo de filas (o columnas) **linealmente independientes**.

En NumPy se calcula con:

```python
rango = np.linalg.matrix_rank(A)
```

**Propiedades clave:**
- Para $A$ de tamaño $m \times n$: $\;\text{rango}(A) \leq \min(m, n)$
- Si todas las filas son independientes → **rango máximo** igual a $\min(m,n)$
- Si alguna fila es combinación lineal de las demás → el rango disminuye

In [37]:
# ── Ejemplo 1: Rango 3 (máximo) ─────────────────────────────────────────
A1 = np.array([[1, 0, 0],
               [0, 2, 0],
               [0, 0, 3]])
print("A1 (columnas independientes):")
print(A1)
print("Rango:", np.linalg.matrix_rank(A1), " ← rango máximo (3)\n")



A1 (columnas independientes):
[[1 0 0]
 [0 2 0]
 [0 0 3]]
Rango: 3  ← rango máximo (3)



#### Comparación del rango de $A$ y $[A \mid b]$

El rango de la **matriz aumentada** $[A \mid b]$ es clave para decidir si un sistema tiene solución.
Veamos un primer ejemplo intuitivo:

In [38]:
A = np.array([[1, 2],
              [3, 6]])          # fila2 = 3·fila1  → rango(A) = 1
b1 = np.array([4, 12])          # consistente: fila2 de b = 3·fila1 de b
b2 = np.array([4, 7])           # inconsistente

Ab1 = np.column_stack([A, b1])
Ab2 = np.column_stack([A, b2])

print("rango(A)          =", np.linalg.matrix_rank(A))
print("rango([A | b1])   =", np.linalg.matrix_rank(Ab1), " (b consistente)")
print("rango([A | b2])   =", np.linalg.matrix_rank(Ab2), " (b inconsistente)")

rango(A)          = 1
rango([A | b1])   = 1  (b consistente)
rango([A | b2])   = 2  (b inconsistente)


---
## Parte 3: ¿Cuántas soluciones tiene el sistema?

Para el sistema $A\mathbf{x} = \mathbf{b}$, con $n$ incógnitas, el
**Teorema de Rouché–Fröbenius** establece:

| Condición | Tipo de sistema |
|-----------|-----------------|
| $\text{rango}(A) \neq \text{rango}([A\mid b])$ | **Sin solución** (incompatible) |
| $\text{rango}(A) = \text{rango}([A\mid b]) = n$ | **Solución única** (compatible determinado) |
| $\text{rango}(A) = \text{rango}([A\mid b]) < n$ | **Infinitas soluciones** (compatible indeterminado) |

Donde $[A\mid b]$ es la **matriz aumentada** (columna $\mathbf{b}$ añadida a $A$).

### Ejemplo A — Sistema $3 \times 3$ con **solución única**

$$\begin{cases}
2x_1 + x_2 - x_3 = 8 \\
-3x_1 - x_2 + 2x_3 = -11 \\
-2x_1 + x_2 + 2x_3 = -3
\end{cases}$$

In [39]:
A_unica = np.array([[ 2,  1, -1],
                    [-3, -1,  2],
                    [-2,  1,  2]], dtype=float)
b_unica = np.array([8, -11, -3], dtype=float)
Ab1 = np.column_stack([A_unica, b_unica])
k=np.linalg.matrix_rank(A_unica)
print("Sistema 3×3 — clasificación:")
n=A_unica.shape[1]
print("Número de columnas (n):", n)
print("Rango de A (k):", k)
print("Rango de [A | b]:", np.linalg.matrix_rank(Ab1))
print("# de variables (n):", n-k)


Sistema 3×3 — clasificación:
Número de columnas (n): 3
Rango de A (k): 3
Rango de [A | b]: 3
# de variables (n): 0


### Ejemplo B — Sistema $3 \times 3$ con **infinitas soluciones**

$$\begin{cases}
x_1 + 2x_2 - x_3 = 3 \\
2x_1 + 4x_2 - 2x_3 = 6 \\
-x_1 - 2x_2 + x_3 = -3
\end{cases}$$

*Las tres ecuaciones son múltiplos de la primera, por lo que no aportan información nueva.*

In [40]:
A_inf = np.array([[ 1,  2, -1],
                  [ 2,  4, -2],   # = 2 × fila 1
                  [-1, -2,  1]],  # = -1 × fila 1
                 dtype=float)
b_inf = np.array([3, 6, -3], dtype=float)   # consistente con los múltiplos
Ab_inf = np.column_stack([A_inf, b_inf])

pk=np.linalg.matrix_rank(A_inf)
print("Sistema 3×3 — clasificación:")
n=A_inf.shape[1]
print("Número de columnas (n):", n)
print("Rango de A (k):", pk)
print("Rango de [A | b]:", np.linalg.matrix_rank(Ab_inf))
print("# de variables (n):", n-pk)

Sistema 3×3 — clasificación:
Número de columnas (n): 3
Rango de A (k): 1
Rango de [A | b]: 1
# de variables (n): 2


### Ejemplo C — Sistema $3 \times 3$ **sin solución**

$$\begin{cases}
x_1 + x_2 + x_3 = 1 \\
x_1 + x_2 + x_3 = 2 \\
x_1 + x_2 + x_3 = 3
\end{cases}$$

*La misma expresión $x_1+x_2+x_3$ no puede valer 1, 2 y 3 al mismo tiempo.*

In [41]:
A_nosol = np.array([[1, 1, 1],
                    [1, 1, 1],
                    [1, 1, 1]], dtype=float)
b_nosol = np.array([1, 2, 3], dtype=float)

Ab_nosol = np.column_stack([A_nosol, b_nosol])

pk=np.linalg.matrix_rank(A_nosol)
print("Sistema 3×3 — clasificación:")
n=A_nosol.shape[1]
print("Número de columnas (n):", n)
print("Rango de A (k):", pk)
print("Rango de [A | b]:", np.linalg.matrix_rank(Ab_nosol))

Sistema 3×3 — clasificación:
Número de columnas (n): 3
Rango de A (k): 1
Rango de [A | b]: 2


---
## Parte 4: Resolver Sistemas de Ecuaciones con NumPy

NumPy ofrece dos funciones principales:

| Función | Cuándo usarla |
|---------|---------------|
| `np.linalg.solve(A, b)` | Sistema **cuadrado** ($m = n$) con **solución única** |
| `np.linalg.lstsq(A, b, rcond=None)` | Sistema **sobredeterminado** ($m > n$) o **mínimos cuadrados** |

> **Regla práctica:** primero clasifica con `clasificar_sistema()`, luego elige la función correcta.

---
### Ejemplo 1: Sistema de 3 ecuaciones — 3 incógnitas

$$\begin{cases}
2x_1 + x_2 - x_3 = 8 \\
-3x_1 - x_2 + 2x_3 = -11 \\
-2x_1 + x_2 + 2x_3 = -3
\end{cases}$$

In [42]:
# ── Paso 1: Definir A y b ────────────────────────────────────────────────
A1 = np.array([[ 2,  1, -1],
               [-3, -1,  2],
               [-2,  1,  2]], dtype=float)
b1 = np.array([8, -11, -3], dtype=float)

Ab1 = np.column_stack([A1, b1])

pk=np.linalg.matrix_rank(A1)
print("Sistema 3×3 — clasificación:")
n=A1.shape[0]
print("Número de filas (n):", n)
print("Rango de A (k):", pk)
print("Rango de [A | b]:", np.linalg.matrix_rank(Ab1))

Sistema 3×3 — clasificación:
Número de filas (n): 3
Rango de A (k): 3
Rango de [A | b]: 3


In [43]:
# ── Paso 2: Resolver con np.linalg.solve() ──────────────────────────────
x1 = np.linalg.solve(A1, b1)

print("\nSolución del sistema A1 x = b1:")
print(x1)


Solución del sistema A1 x = b1:
[ 2.  3. -1.]


In [44]:
# ── Paso 3: Verificar — A·x debe ser igual a b ──────────────────────────
residuo = A1 @ x1 - b1
print("Verificación  A·x - b =", residuo)
print("(Todos los valores deben ser ≈ 0)")

Verificación  A·x - b = [-8.88178420e-16  1.77635684e-15  8.88178420e-16]
(Todos los valores deben ser ≈ 0)


---
### Ejemplo 2: Sistema de 5 ecuaciones — 4 incógnitas

Un sistema con **más ecuaciones que incógnitas** ($m > n$) se llama **sobredeterminado**.
Puede ocurrir que:
- Algunas ecuaciones sean redundantes → **hay solución exacta**
- Las ecuaciones sean inconsistentes → **no hay solución exacta**, se usa mínimos cuadrados

#### Caso 2a — consistente (solución exacta)

$$\begin{cases}
x_1 + x_2 \phantom{+ x_3 + x_4} = 3 \\
\phantom{x_1 +{}} x_2 + x_3 \phantom{+ x_4} = 5 \\
\phantom{x_1 + x_2 +{}} x_3 + x_4 = 7 \\
x_1 \phantom{+ x_2 + x_3} + x_4 = 5 \\
x_1 + x_2 + x_3 + x_4 = 10
\end{cases}$$

In [45]:
# ── Caso 2a: 5 ecuaciones, 4 incógnitas — sistema consistente ───────────
A2a = np.array([[1, 1, 0, 0],
                [0, 1, 1, 0],
                [0, 0, 1, 1],
                [1, 0, 0, 1],
                [1, 1, 1, 1]], dtype=float)
b2a = np.array([3, 5, 7, 5, 10], dtype=float)

Ab2a = np.column_stack([A2a, b2a])

pk=np.linalg.matrix_rank(A2a)
print("Sistema 5×4 — clasificación:")
n=A2a.shape[0]
print("Número de filas (n):", n)
print("Rango de A (k):", pk)
print("Rango de [A | b]:", np.linalg.matrix_rank(Ab2a))

Sistema 5×4 — clasificación:
Número de filas (n): 5
Rango de A (k): 3
Rango de [A | b]: 3


#### Caso 2b — inconsistente (sin solución exacta → mínimos cuadrados)

Modificamos el último término independiente para que las ecuaciones sean contradictorias:

In [46]:
# ── Caso 2b: mismo A, pero b inconsistente ──────────────────────────────
A2b = A2a.copy()
b2b = np.array([3, 5, 7, 5, 11], dtype=float)   # última ecuación ≠ suma de las demás

print("=== Caso 2b: 5 ecuaciones, 4 incógnitas (inconsistente) ===")
clasificar_sistema(A2b, b2b)
print()

# lstsq minimiza ||A·x - b||² → solución de mínimos cuadrados
x2b, _, _, _ = np.linalg.lstsq(A2b, b2b, rcond=None)

print("Solución de mínimos cuadrados (minimiza el error cuadrático):")
for i, xi in enumerate(x2b, start=1):
    print(f"  x{i} = {xi:.4f}")

error2b = np.linalg.norm(A2b @ x2b - b2b)
print(f"\nError residual ||A·x - b|| = {error2b:.6f}")

=== Caso 2b: 5 ecuaciones, 4 incógnitas (inconsistente) ===


NameError: name 'clasificar_sistema' is not defined

---
### Ejemplo 3: Sistema de 6 ecuaciones — 6 incógnitas

Sistema **tridiagonal** (modelo de temperatura en una barra con extremos fijos a 100°C):

$$A = \begin{pmatrix}
 4 & -1 &  0 &  0 &  0 &  0 \\
-1 &  4 & -1 &  0 &  0 &  0 \\
 0 & -1 &  4 & -1 &  0 &  0 \\
 0 &  0 & -1 &  4 & -1 &  0 \\
 0 &  0 &  0 & -1 &  4 & -1 \\
 0 &  0 &  0 &  0 & -1 &  4
\end{pmatrix}, \qquad
\mathbf{b} = \begin{pmatrix} 100 \\ 0 \\ 0 \\ 0 \\ 0 \\ 100 \end{pmatrix}$$

In [ ]:
# ── Sistema 6×6 tridiagonal ───────────────────────────────────────────────
A6 = np.array([[ 4, -1,  0,  0,  0,  0],
               [-1,  4, -1,  0,  0,  0],
               [ 0, -1,  4, -1,  0,  0],
               [ 0,  0, -1,  4, -1,  0],
               [ 0,  0,  0, -1,  4, -1],
               [ 0,  0,  0,  0, -1,  4]], dtype=float)
b6 = np.array([100, 0, 0, 0, 0, 100], dtype=float)

print("=== Sistema 6×6 ===")
clasificar_sistema(A6, b6)
print()

In [ ]:
x6 = np.linalg.solve(A6, b6)

print("Solución (temperaturas internas de la barra):")
for i, xi in enumerate(x6, start=1):
    print(f"  x{i} = {xi:.6f}")

print(f"\nVerificación  ||A·x - b|| = {np.linalg.norm(A6 @ x6 - b6):.2e}")

---
### Ejemplo 4: Sistema $10 \times 10$ generado con NumPy

Para sistemas grandes se usa exactamente la misma instrucción.
Construimos un sistema $N \times N$ con solución conocida
$\mathbf{x}^* = (1, 2, 3, \ldots, N)^T$ y verificamos que NumPy la recupera:

In [ ]:
# ── Sistema 10×10 con solución conocida ─────────────────────────────────
N = 10
np.random.seed(42)

# Matriz aleatoria con diagonal dominante (garantiza invertibilidad)
A_10 = np.random.randint(1, 6, (N, N)).astype(float)
for i in range(N):
    A_10[i, i] += N * 5          # aumentar diagonal

x_real_10 = np.arange(1, N + 1, dtype=float)   # [1, 2, ..., 10]
b_10 = A_10 @ x_real_10

print(f"=== Sistema {N}×{N} ===")
clasificar_sistema(A_10, b_10)
print()

x_sol_10 = np.linalg.solve(A_10, b_10)
print("Solución calculada por NumPy:")
print(np.round(x_sol_10, 6))
print("\nSolución real conocida:")
print(x_real_10)
print(f"\nError máximo |x_real - x_calculado| = {np.max(np.abs(x_sol_10 - x_real_10)):.2e}")

### Escalando a sistemas aún más grandes

La ventaja de NumPy es que funciona igual para cualquier tamaño $N$.
Observa el tiempo de resolución y el error numérico al aumentar $N$:

In [ ]:
import time

for N in [20, 50, 100, 500]:
    np.random.seed(0)
    A_n = np.random.rand(N, N) + N * np.eye(N)   # diagonal dominante
    x_n = np.arange(1, N + 1, dtype=float)
    b_n = A_n @ x_n

    t0 = time.time()
    x_calc = np.linalg.solve(A_n, b_n)
    dt = time.time() - t0

    error = np.max(np.abs(x_calc - x_n))
    rango = np.linalg.matrix_rank(A_n)
    print(f"N = {N:4d} | rango = {rango:4d} | Error máx = {error:.2e} | Tiempo = {dt*1000:.2f} ms")

---
## Ejercicios Propuestos

### Ejercicio 1 — Sistema $3 \times 3$

**1a)** Clasifica y resuelve (si es posible) el siguiente sistema:

$$\begin{cases}
x_1 - 2x_2 + 3x_3 = 9 \\
-x_1 + 3x_2 \phantom{+ 3x_3} = -4 \\
2x_1 - 5x_2 + 5x_3 = 17
\end{cases}$$

**1b)** Verifica tu solución calculando $A\mathbf{x} - \mathbf{b}$.

In [ ]:
# Ejercicio 1 — escribe tu solución aquí
A_ej1 = np.array([[ 1, -2,  3],
                  [-1,  3,  0],
                  [ 2, -5,  5]], dtype=float)
b_ej1 = np.array([9, -4, 17], dtype=float)

# 1. Clasifica el sistema
# TODO: clasificar_sistema(A_ej1, b_ej1)

# 2. Resuelve el sistema
# TODO: x_ej1 = np.linalg.solve(A_ej1, b_ej1)

# 3. Verifica: A_ej1 @ x_ej1 - b_ej1 ≈ 0


### Ejercicio 2 — Sistema $4 \times 4$ con infinitas soluciones

Verifica que el siguiente sistema tiene **infinitas soluciones** y determina
el número de **grados de libertad**:

$$\begin{cases}
x_1 + x_2 + x_3 + x_4 = 4 \\
x_1 - x_2 + x_3 - x_4 = 0 \\
2x_1 \phantom{- x_2} + 2x_3 \phantom{- x_4} = 4 \\
\phantom{x_1 - {}}2x_2 \phantom{+ x_3 -} + 2x_4 = 4
\end{cases}$$

*Pista: observa si alguna fila de $A$ es combinación lineal de las demás.*

In [ ]:
# Ejercicio 2 — escribe tu solución aquí
A_ej2 = np.array([[1,  1, 1,  1],
                  [1, -1, 1, -1],
                  [2,  0, 2,  0],
                  [0,  2, 0,  2]], dtype=float)
b_ej2 = np.array([4, 0, 4, 4], dtype=float)

# TODO: clasifica el sistema y describe los grados de libertad


### Ejercicio 3 — Sistema sobredeterminado $6 \times 4$

El siguiente sistema tiene 6 ecuaciones y 4 incógnitas con **pequeño ruido** en los datos.
¿Tiene solución exacta? De no ser así, encuentra la **solución de mínimos cuadrados**
y calcula el error residual.

In [ ]:
# Ejercicio 3 — escribe tu solución aquí
np.random.seed(7)
A_ej3 = np.random.randint(-3, 4, (6, 4)).astype(float)
x_verdadero = np.array([1.0, -2.0, 3.0, -1.0])
b_ej3 = A_ej3 @ x_verdadero + np.random.randn(6) * 0.5   # pequeño ruido

print("A =")
print(A_ej3)
print("\nb =", np.round(b_ej3, 4))

# 1. Clasifica el sistema
# TODO: clasificar_sistema(A_ej3, b_ej3)

# 2. Solución de mínimos cuadrados
# TODO: x_ej3, _, _, _ = np.linalg.lstsq(A_ej3, b_ej3, rcond=None)

# 3. Compara con x_verdadero = [1, -2, 3, -1]
# TODO: print("Error:", np.linalg.norm(x_ej3 - x_verdadero))


### Ejercicio 4 — Sistema $8 \times 8$

Resuelve el siguiente sistema $8 \times 8$ y verifica el resultado.
La solución real es $\mathbf{x}^* = (2, 4, 6, 8, 10, 12, 14, 16)^T$.

In [ ]:
# Ejercicio 4 — escribe tu solución aquí
np.random.seed(99)
A_ej4 = 5 * np.eye(8) + np.random.rand(8, 8)
x_ej4_real = np.array([2, 4, 6, 8, 10, 12, 14, 16], dtype=float)
b_ej4 = A_ej4 @ x_ej4_real

print("b =", np.round(b_ej4, 4))

# 1. Clasifica el sistema
# TODO

# 2. Resuelve
# TODO: x_ej4 = np.linalg.solve(A_ej4, b_ej4)

# 3. Error
# TODO: print("Error:", np.max(np.abs(x_ej4 - x_ej4_real)))


---
## Resumen

| Concepto | Función NumPy | Código ejemplo |
|----------|--------------|----------------|
| Crear matriz | `np.array([[...]])` | `A = np.array([[1,2],[3,4]])` |
| Rango | `np.linalg.matrix_rank(A)` | `r = np.linalg.matrix_rank(A)` |
| Matriz aumentada | `np.column_stack([A, b])` | `Ab = np.column_stack([A, b])` |
| Resolver (cuadrado) | `np.linalg.solve(A, b)` | `x = np.linalg.solve(A, b)` |
| Mínimos cuadrados | `np.linalg.lstsq(A, b, rcond=None)` | `x, _, _, _ = np.linalg.lstsq(A, b, rcond=None)` |

**Flujo de trabajo recomendado:**

```
1. Definir A y b
2. clasificar_sistema(A, b)   ← ¿cuántas soluciones hay?
3. Si tiene solución única y A es cuadrada → np.linalg.solve(A, b)
   Si el sistema es sobredeterminado       → np.linalg.lstsq(A, b, rcond=None)
4. Verificar: A @ x - b ≈ 0
```